# 02. プロンプトエンジニアリング：few-shot・chain-of-thought

## はじめに

**プロンプトエンジニアリング**とは、LLMから望む出力を得るためのプロンプト設計技術です。
同じモデルでも、プロンプトの書き方次第で出力品質が大きく変わります。

このノートブックで学ぶこと：
- **Zero-shot vs. Instructed**: 指示の有無による違い
- **Few-shot プロンプティング**: 例示による精度向上
- **Chain-of-Thought（CoT）**: 推論ステップを促す技法
- **システムプロンプト**: ペルソナ・役割の設定
- **プロンプトテンプレート**: 再利用可能なテンプレート設計

**参考文献**:
- [Prompt Engineering Guide](https://www.promptingguide.ai/ja)
- Wei et al. (2022) - Chain-of-Thought Prompting

In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ollama_client import OllamaClient, get_client

# デフォルトモデル
MODEL = "qwen2.5:7b"

client = get_client(model=MODEL)

if not client.is_available():
    raise RuntimeError("Ollamaサーバーが起動していません。'ollama serve' を実行してください。")

print(f"モデル: {MODEL}")
print("準備完了")

# ヘルパー関数: プロンプトと応答を見やすく表示
def show_result(label, prompt, response):
    print(f"\n{'='*50}")
    print(f"[{label}]")
    print(f"プロンプト:\n{prompt}")
    print(f"\n応答:\n{response}")
    print('='*50)

## 1. 基本的なプロンプト設計

**Zero-shot** は例示なしで直接質問する方法です。
**Instructed** は明示的に指示を与えることで出力形式や品質を改善します。

In [ ]:
# Zero-shot: 指示なしで直接質問
zero_shot = "機械学習とは"
response_zero = client.generate(zero_shot)
show_result("Zero-shot", zero_shot, response_zero)

# Instructed: 明確な指示を与える
instructed = """機械学習とは何かを、以下の条件で説明してください：
- 対象: プログラミング未経験の高校生
- 長さ: 3文以内
- 難しい専門用語は使わない
- 身近な例えを使う"""
response_instructed = client.generate(instructed)
show_result("Instructed", instructed, response_instructed)

print("\n観察: 指示を具体的にするほど、期待通りの出力が得られます")

## 2. Few-shotプロンプティング

**Few-shot** は、プロンプト内に入出力の例を含めることで、
モデルにタスクのパターンを学習させる技法です。
特に分類・変換タスクで効果的です。

In [ ]:
# Few-shot: 感情分析の例
few_shot_sentiment = """テキストの感情を「ポジティブ」「ネガティブ」「中立」のいずれかで分類してください。

例:
テキスト: 「今日は最高の天気で気分が上がった！」
感情: ポジティブ

テキスト: 「電車が遅延して大事な会議に遅刻してしまった。最悪だ。」
感情: ネガティブ

テキスト: 「明日の会議は午後3時から始まります。」
感情: 中立

では、以下を分類してください：
テキスト: 「新しいプロジェクトは難しいけど、チームと一緒に取り組むのが楽しい。」
感情:"""

response_sentiment = client.generate(few_shot_sentiment, temperature=0.1)
show_result("Few-shot 感情分析", few_shot_sentiment, response_sentiment)

# 複数のテキストをバッチ処理
test_texts = [
    "このレストランの料理は普通だった。",
    "バグが全部直って本当によかった！リリースできる！",
    "プロジェクトの締め切りは来週金曜日です。",
]

print("\n=== バッチ感情分析 ===")
for text in test_texts:
    prompt = f"""テキストの感情を「ポジティブ」「ネガティブ」「中立」で答えてください（1単語のみ）。

例: 「今日は最高！」→ ポジティブ
例: 「最悪な一日だった」→ ネガティブ
例: 「会議は3時です」→ 中立

テキスト: 「{text}」
感情:"""
    result = client.generate(prompt, temperature=0.0)
    print(f"  '{text}'")
    print(f"  → {result.strip()}\n")

In [ ]:
# Few-shot: 翻訳スタイルの統一
few_shot_translation = """以下の英語のIT用語を、カジュアルな日本語のSNS投稿風に翻訳してください。

例:
英語: "I just deployed the new feature to production!"
日本語: 「新機能を本番環境にデプロイしたよ！ドキドキ✨」

例:
英語: "The API is returning a 500 error."
日本語: 「APIが500エラー返してる…😱 なんで〜」

例:
英語: "We finally merged the pull request after 3 weeks of review."
日本語: 「3週間かかったPRがようやくマージされたー！🎉 長かった…」

翻訳してください:
英語: "I spent all night debugging and finally found the bug. It was just a missing semicolon."
日本語:"""

response_translation = client.generate(few_shot_translation)
show_result("Few-shot 翻訳（スタイル統一）", few_shot_translation, response_translation)

## 3. Chain-of-Thought（CoT）推論

**Chain-of-Thought** は、モデルに思考過程を段階的に示させることで、
複雑な推論タスクの精度を向上させる技法です。
「ステップごとに考えてください」という一文が大きな効果を持ちます。

In [ ]:
# 数学の文章題
math_problem = """田中さんは本屋で本を3冊買いました。
1冊目は1200円、2冊目は980円、3冊目は1冊目の半額でした。
5000円で支払った場合、おつりはいくらですか？"""

# CoTなし
prompt_no_cot = f"{math_problem}\n\n答え（数字のみ）:"
response_no_cot = client.generate(prompt_no_cot, temperature=0.0)
show_result("CoTなし", prompt_no_cot, response_no_cot)

# CoTあり
prompt_with_cot = f"{math_problem}\n\nステップごとに考えてください。"
response_with_cot = client.generate(prompt_with_cot, temperature=0.0)
show_result("CoTあり", prompt_with_cot, response_with_cot)

In [ ]:
# 論理推論の比較
logic_problem = """AはBより年上です。CはAより年上です。BはDより年上です。
最も年下なのは誰ですか？"""

print("=== 論理推論タスク ===")
print(f"問題: {logic_problem}")
print()

# 直接回答
direct_response = client.generate(
    logic_problem + "\n答え（1文字のみ）:",
    temperature=0.0
)
print(f"[直接回答] {direct_response.strip()}")

# CoT回答
cot_response = client.generate(
    logic_problem + "\n\nそれぞれの年齢関係を整理して、ステップごとに考えてください。",
    temperature=0.0
)
print(f"\n[CoT回答]\n{cot_response}")

print("\n観察: CoTにより、モデルが推論過程を明示的に示すことで精度が向上します")

## 4. システムプロンプトの活用

**システムプロンプト** は、モデルのペルソナ・役割・制約を事前に設定するものです。
会話全体を通じて一貫した振る舞いを実現できます。

In [ ]:
# システムプロンプトでペルソナを設定
personas = [
    {
        "name": "厳格な先生",
        "system": "あなたは厳格な大学教授です。学術的に正確で、曖昧な点は明確に指摘し、専門用語を使って説明します。"
    },
    {
        "name": "フレンドリーな友人",
        "system": "あなたはITに詳しい親友です。タメ口で、絵文字を使って、例え話を交えながら楽しく説明します。"
    },
    {
        "name": "ビジネスコンサルタント",
        "system": "あなたはビジネスコンサルタントです。ROIとビジネス価値を重視し、具体的な数値と事例を用いて説明します。"
    },
]

question = "AIを使ったビジネス活用について教えてください。"

print(f"質問: {question}")
print("=" * 60)

for persona in personas:
    messages = [
        {"role": "system", "content": persona["system"]},
        {"role": "user", "content": question}
    ]
    response = client.chat(messages)
    print(f"\n[{persona['name']}]")
    print(response[:300] + ("..." if len(response) > 300 else ""))
    print("-" * 40)

## 5. プロンプトテンプレート

プロンプトテンプレートを使うと、再利用可能で保守しやすいコードが書けます。
Pythonのf-stringやクラスを使って構造化したテンプレートを設計しましょう。

In [ ]:
# プロンプトテンプレート関数

def qa_template(context: str, question: str, language: str = "日本語") -> str:
    """Q&Aタスク用プロンプトテンプレート"""
    return f"""以下のコンテキストを参考に、質問に{language}で答えてください。
コンテキスト外の情報を使う場合は、その旨を明記してください。

コンテキスト:
{context}

質問: {question}

回答:"""


def classification_template(categories: list, text: str, examples: list = None) -> str:
    """分類タスク用プロンプトテンプレート"""
    categories_str = "、".join(f'「{c}」' for c in categories)
    
    examples_str = ""
    if examples:
        examples_str = "\n\n例:\n"
        for ex in examples:
            examples_str += f"テキスト: {ex['text']}\nカテゴリ: {ex['label']}\n\n"
    
    return f"""以下のテキストを{categories_str}のいずれかに分類してください。
カテゴリ名のみ回答してください。{examples_str}
テキスト: {text}
カテゴリ:"""


def summarize_template(text: str, max_chars: int = 200, style: str = "箇条書き") -> str:
    """要約タスク用プロンプトテンプレート"""
    return f"""以下のテキストを{max_chars}字以内で{style}形式に要約してください。

テキスト:
{text}

要約:"""


# テンプレートの使用例を表示
sample_context = """
東京は日本の首都で、世界最大の都市圏の一つです。
人口は約1400万人（都市圏では3700万人以上）。
経済、文化、テクノロジーの中心地であり、多数のフォーチュン500企業が本社を置いています。
"""

prompt = qa_template(sample_context, "東京の人口はどれくらいですか？")
print("=== Q&Aテンプレート ===")
print("生成されたプロンプト:")
print(prompt)
print("\n応答:")
response = client.generate(prompt, temperature=0.1)
print(response)

In [ ]:
# 分類テンプレートの実用例
print("=== 分類テンプレート ===")

categories = ["技術", "ビジネス", "スポーツ", "エンタメ", "政治"]
examples = [
    {"text": "新しいiPhoneが発売された", "label": "技術"},
    {"text": "日本代表がワールドカップ出場決定", "label": "スポーツ"},
]

test_texts = [
    "AIスタートアップが100億円の資金調達を実施",
    "人気アイドルグループが解散を発表",
    "国会で新しい経済対策が審議される",
    "量子コンピュータの実用化に向けた研究が進む",
]

print(f"カテゴリ: {categories}\n")
for text in test_texts:
    prompt = classification_template(categories, text, examples)
    result = client.generate(prompt, temperature=0.0)
    print(f"'{text}'")
    print(f"  → {result.strip()}\n")

# 要約テンプレートの実用例
print("\n=== 要約テンプレート ===")

long_text = """
人工知能（AI）技術の急速な発展により、様々な産業でデジタルトランスフォーメーション（DX）が加速しています。
特に自然言語処理（NLP）分野では、大規模言語モデル（LLM）の登場により、
テキスト生成・翻訳・要約・コード生成など多様なタスクが自動化されつつあります。
企業はこれらの技術を活用して業務効率化を図る一方、
著作権・プライバシー・雇用への影響など倫理的・社会的課題への対応も求められています。
日本政府もAI戦略を策定し、産学官連携でAI人材育成や規制整備を進めています。
"""

prompt = summarize_template(long_text, max_chars=150, style="箇条書き")
response = client.generate(prompt)
print("要約結果:")
print(response)

## まとめ

このノートブックで学んだプロンプトエンジニアリングの技法：

| 技法 | 特徴 | 効果的な場面 |
|------|------|-------------|
| Zero-shot | 例なし・直接質問 | シンプルなタスク |
| Instructed | 明確な指示を与える | 出力形式を制御したいとき |
| Few-shot | 入出力例を含める | 分類・変換・スタイル統一 |
| Chain-of-Thought | 推論ステップを促す | 複雑な推論・数学・論理 |
| システムプロンプト | ペルソナ・役割設定 | 一貫したキャラクターが必要な会話 |
| テンプレート | 再利用可能な構造 | 同じパターンを繰り返すタスク |

**プロンプト設計の原則**：
1. **明確さ**: タスクを具体的に指示する
2. **コンテキスト**: 必要な背景情報を提供する
3. **例示**: 期待する入出力のパターンを示す
4. **制約**: 出力形式・長さ・スタイルを指定する
5. **反復改善**: 試行錯誤でプロンプトを改善する

**次のステップ**：
- `03_rag_from_scratch.ipynb` で埋め込みとRAGパイプラインを実装する